In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.LinearRegression.build_x import add_gene_signature
from metab_processing.LinearRegression.least_squares import (
    fit_gene_betas, subsample_gene_betas, rank_coefficients, plot_beta_histogram)

PATH = f'{DATA_DIR}/Xenium/Primary_Dermal_Melanoma'
TOP_N = 'all'
ANNOT_COL, CELL_TYPE = 'Tier1', 'T Cell'
METHOD, PENALTY, STANDARDIZE = 'l1', 0.001, True
N_SUB = 200
SIG = 'Tcyto_vs_exhaustion'          # signature name
POSITIVE = ['CD8A', 'GZMB', 'IFNG']  # <- set to genes in adata.var_names / uns['x_genes']
NEGATIVE = ['PDCD1', 'LAG3']

In [ ]:
adata = sc.read_h5ad(Path(PATH) / 'LinearRegression' / f'hvg{TOP_N}_x_adata.h5ad')

In [ ]:
adata = add_gene_signature(adata, SIG, positive=POSITIVE, negative=NEGATIVE)  # factor_mode='union'

In [ ]:
betas = fit_gene_betas(adata, genes=[SIG], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                       method=METHOD, penalty=PENALTY, standardize=STANDARDIZE)

In [ ]:
sub = subsample_gene_betas(adata, SIG, factors=None, n_subsamples=N_SUB,
                           annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                           method=METHOD, penalty=PENALTY, standardize=STANDARDIZE)

In [ ]:
def add_subsample_percentiles(betas_df, sub, lower=2.5, upper=97.5):
    """Add min/max and lower/upper-percentile subsample-beta columns to betas_df,
    matched by factor (`sub` = n_subsamples x factor DataFrame from subsample_gene_betas)."""
    out = betas_df.copy()
    def stat(fn):
        return [fn(sub[f]) if f in sub.columns else np.nan for f in out['factor']]
    out['min_subsample_beta'] = stat(np.nanmin)
    out['max_subsample_beta'] = stat(np.nanmax)
    out[f'lower_{lower}_subsample_beta'] = stat(lambda s: np.nanpercentile(s, lower))
    out[f'upper_{upper}_subsample_beta'] = stat(lambda s: np.nanpercentile(s, upper))
    return out

In [ ]:
display(add_subsample_percentiles(rank_coefficients(betas), sub).head(30))

In [ ]:
# Distribution of one factor's beta across the resamples.
FACTOR = sub.columns[0]   # <- set to any factor (tf / lig$rec / lig#tf / metab@name)
plot_beta_histogram(sub[FACTOR], title=f'{SIG} ~ {FACTOR}');